# Weekly project 6
Today we will continue work from monday.
We will follow the style of last week.

Weekly project:
- You will need to implement your own k-means algorithm. You are not allowed to use the implementation in `sklearn`.
- It should be able to cluster each of the different figures.
- Extend your k-means so it finds the optimal amount of clusters.
  
## Challenge
- Implement the mean shift clustering algorithm


In [2]:
import numpy as np
import open3d as o3d
import copy
import matplotlib.pyplot as plt

def draw_labels_on_model(pcl, labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    o3d.visualization.draw_geometries([pcl_temp])

d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_sphere().translate((0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_cone().translate((d, -d, 0))

# sample points
point_cloud = mesh.sample_points_uniformly(int(1e3))


In [4]:
import numpy as np

def kmeans(X, k, max_iter=100, tol=1e-4, random_state=0):
    """
    Pure NumPy K-means.
    Returns: labels (n,), centers (k,d), inertia (float)
    """
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    rng = np.random.default_rng(random_state)

    # --- k-means++ init ---
    centers = np.empty((k, d))
    centers[0] = X[rng.integers(n)]
    dist2 = np.full(n, np.inf)
    for i in range(1, k):
        dist2 = np.minimum(dist2, ((X - centers[i-1])**2).sum(axis=1))
        probs = dist2 / dist2.sum()
        centers[i] = X[rng.choice(n, p=probs)]

    # --- Lloyd's iterations ---
    for _ in range(max_iter):
        # assign
        d2 = ((X[:, None, :] - centers[None, :, :])**2).sum(axis=2)  # (n,k)
        labels = d2.argmin(axis=1)
        # update
        new_centers = np.vstack([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centers[j]
            for j in range(k)
        ])
        shift = np.linalg.norm(new_centers - centers)
        centers = new_centers
        if shift < tol:
            break

    inertia = np.take_along_axis(
        ((X[:, None, :] - centers[None, :, :])**2).sum(axis=2),
        labels[:, None], axis=1
    ).sum()
    return labels, centers, inertia


def auto_kmeans(X, k_min=2, k_max=12, random_state=0):
    """
    Picks K by simple elbow on inertia (first knee found).
    Returns: best_k, labels, centers
    """
    X = np.asarray(X)
    inertias, results = [], []
    for k in range(k_min, k_max+1):
        labels, centers, inertia = kmeans(X, k, random_state=random_state)
        inertias.append(inertia); results.append((k, labels, centers))
    inertias = np.array(inertias)

    # knee via maximum second derivative on log-scale
    y = np.log(inertias)
    curv = np.zeros_like(y)
    curv[1:-1] = y[:-2] - 2*y[1:-1] + y[2:]
    i_best = np.argmax(curv[1:-1]) + 1  # avoid edges
    best_k, best_labels, best_centers = results[i_best]
    return best_k, best_labels, best_centers


In [6]:
# points only (you can also concatenate normals if you want)
X = np.asarray(point_cloud.points)

# fixed-K example
labels, centers, _ = kmeans(X, k=6, random_state=0)
draw_labels_on_model(point_cloud, labels.astype(int))


